# Evolutionary Deep Nets

Compare dense and sparse models. Use `SET_RUN_MODE=full` for full training.


In [ ]:
import os
from pathlib import Path
import time
import pickle
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.regularizers import Regularizer, l1, l2
import matplotlib.pyplot as plt

# Smoke mode runs every model/target on a small subset. Full mode uses the
# original sample split and 50 epochs (200 for SET DNN).
RUN_MODE = os.environ.get("SET_RUN_MODE", "smoke")
assert RUN_MODE in {"smoke", "full"}
EPOCHS = 1 if RUN_MODE == "smoke" else 50
SET_DNN_EPOCHS = 2 if RUN_MODE == "smoke" else 200
SET_EPOCHS = 2 if RUN_MODE == "smoke" else 50
keras.utils.set_random_seed(42)
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Mode:", RUN_MODE, "TensorFlow:", tf.__version__)


In [ ]:
# Shared data; override with SET_DATA_DIR.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "adaptive_sparsity/research_set.py").is_file())
DATA_DIR = Path(os.environ.get("SET_DATA_DIR", str(ROOT / "data and requirements")))


## Load data


In [ ]:
# Both CSVs contain a header and a nonnumeric row-name column.
train = np.loadtxt(DATA_DIR / "samX.csv", delimiter=",", skiprows=1,
                   usecols=range(1, 601), dtype=np.float32)
label = np.loadtxt(DATA_DIR / "samY.csv", delimiter=",", skiprows=1,
                   usecols=range(1, 5), dtype=np.float32)
assert len(train) == len(label)
assert np.isfinite(train).all() and np.isfinite(label).all()
assert np.isin(label, [0, 1]).all()


In [ ]:
train=np.array(train)
label=np.array(label)
train=train.astype('float32')
label=label.astype('float32')
print(train.shape)
print(label.shape)

In [ ]:
train = train.reshape((len(train), train.shape[1], 1))
print(train.shape)


In [ ]:
# Preserve the original ordered split: first 2,000 held out.
trainx, trainy = train[2000:], label[2000:]
testx, testy = train[:2000], label[:2000]
if RUN_MODE == "smoke":
    trainx, trainy = trainx[:128], trainy[:128]
    testx, testy = testx[:64], testy[:64]
testylis = [testy[:, i:i+1] for i in range(4)]
print("Training:", trainx.shape, trainy.shape, "Test:", testx.shape, testy.shape)


In [ ]:
lbs = [trainy[:, i:i+1] for i in range(4)]


## Sparse functions


In [ ]:
epsilon = 20
zeta = 0.3

@keras.utils.register_keras_serializable(package="SET")
class MaskWeights(keras.constraints.Constraint):
    def __init__(self, mask):
        self.mask = np.asarray(mask, dtype=np.float32)
    def __call__(self, w):
        return w * tf.cast(self.mask, w.dtype)
    def get_config(self):
        return {"mask": self.mask.tolist()}

def rewireMask(weights, noWeights):
    # Prune weights nearest zero separately by sign, then restore edge count.
    flat = weights.ravel()
    core = (flat != 0)
    for indices in (np.flatnonzero(flat < 0), np.flatnonzero(flat > 0)):
        count = int(zeta * len(indices))
        if count:
            removed = indices[np.argsort(np.abs(flat[indices]))[:count]]
            core[removed] = False
    mask = core.copy()
    count = int(noWeights) - int(mask.sum())
    assert 0 <= count <= int((~mask).sum())
    if count:
        mask[np.random.choice(np.flatnonzero(~mask), count, replace=False)] = True
    return mask.reshape(weights.shape).astype('float32'), core.reshape(weights.shape).astype('float32')

def createWeightsMask(epsilon, noRows, noCols):
    probability = min(1., epsilon * (noRows + noCols) / (noRows * noCols))
    mask = (np.random.random((noRows, noCols)) < probability).astype('float32')
    return int(mask.sum()), mask


## Train models


In [ ]:
def count_neurons(model):
    return sum(int(np.count_nonzero(np.sum(np.abs(w[0]), axis=1) > 1e-3))
               for layer in model.layers
               if len(w := layer.get_weights()) and w[0].ndim == 2)

class TrainHistory(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.losses = []
        self.neurons = [count_neurons(self.model)]
    def on_train_batch_end(self, batch, logs=None):
        self.losses.append((logs or {}).get('loss'))
        self.neurons.append(count_neurons(self.model))

@keras.utils.register_keras_serializable(package="SET")
class L21(Regularizer):
    def __init__(self, C=0.):
        self.C = float(C)
    def __call__(self, x):
        # Epsilon avoids undefined gradients for an all-zero row.
        return self.C * tf.sqrt(tf.cast(tf.shape(x)[1], x.dtype)) * tf.reduce_sum(
            tf.sqrt(tf.reduce_sum(tf.square(x), axis=1) + keras.backend.epsilon()))
    def get_config(self):
        return {'C': self.C}


In [ ]:
# normal DNN model

def create_nonconv_model(windowsize):
  li=layers.Input((windowsize,1))
  x=layers.BatchNormalization()(li)
  x=layers.Flatten()(x)
  x=layers.Dropout(0.2)(x)
  output=layers.Dense(100)(x)
  output=layers.Dense(1,activation='sigmoid')(output)
  model=models.Model(inputs=li, outputs=output)
  return model

DNNmodel=[]
DNNmodel.append(create_nonconv_model(600))
DNNmodel.append(create_nonconv_model(600))
DNNmodel.append(create_nonconv_model(600))
DNNmodel.append(create_nonconv_model(600))
DNNhis=[]
start_time = time.time()
for i in range(4):

  DNNmodel[i].compile(optimizer='sgd',loss='mean_squared_error',metrics= ['accuracy','mean_absolute_error',tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.Precision(name='precision')])
  DNNhis.append(DNNmodel[i].fit(trainx,lbs[i],epochs=EPOCHS))
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
lambda_val = .7

# normal las DNN model

def create_nonconv_lasso_model(windowsize):
  li=layers.Input((windowsize,1))
  x=layers.BatchNormalization()(li)
  x=layers.Flatten()(x)
  x=layers.Dropout(0.2)(x)
  output=layers.Dense(100)(x)
  output=layers.Dense(1,activation='sigmoid',kernel_regularizer = l1(lambda_val))(output)
  model=models.Model(inputs=li, outputs=output)
  return model

DNNlasmodel=[]
DNNlasmodel.append(create_nonconv_lasso_model(600))
DNNlasmodel.append(create_nonconv_lasso_model(600))
DNNlasmodel.append(create_nonconv_lasso_model(600))
DNNlasmodel.append(create_nonconv_lasso_model(600))
DNNlashis=[]
start_time = time.time()
for i in range(4):

  DNNlasmodel[i].compile(optimizer='adam',loss='mean_squared_error',metrics= ['accuracy','mean_absolute_error',tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.Precision(name='precision')])
  DNNlashis.append(DNNlasmodel[i].fit(trainx,lbs[i],epochs=EPOCHS))

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:

# normal group las DNN model

def create_nonconv_group_lasso_model(windowsize):
  li=layers.Input((windowsize,1))
  x=layers.BatchNormalization()(li)
  x=layers.Flatten()(x)
  x=layers.Dropout(0.2)(x)
  output=layers.Dense(100)(x)
  output=layers.Dense(1,activation='sigmoid',kernel_regularizer=L21(0.001))(output)
  model=models.Model(inputs=li, outputs=output)
  return model

DNNgrouplasmodel=[]
DNNgrouplasmodel.append(create_nonconv_group_lasso_model(600))
DNNgrouplasmodel.append(create_nonconv_group_lasso_model(600))
DNNgrouplasmodel.append(create_nonconv_group_lasso_model(600))
DNNgrouplasmodel.append(create_nonconv_group_lasso_model(600))
DNNgrouplashis=[]
nehis=[]
start_time = time.time()
for i in range(4):

  DNNgrouplasmodel[i].compile(optimizer='adam',loss='mean_squared_error',metrics= ['accuracy','mean_absolute_error',tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.Precision(name='precision')])
  DNNgrouplashis.append(DNNgrouplasmodel[i].fit(trainx,lbs[i],epochs=EPOCHS))
  nehis.append(TrainHistory())

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
# SET DNN
def weightsEvolution(model):
# this represents the core of the SET procedure. It removes the weights closest to zero in each layer and add new random weights

    bat=model.get_layer("bat_layer").get_weights()
    w2=model.get_layer("dense_layer2").get_weights()
    w1 = model.get_layer("dense_layer1").get_weights()
    
    [wm1, wm1Core] = rewireMask(w1[0], noPar1)

    w1[0] = w1[0] * wm1Core
    return bat,w1,w2,wm1

def create_SET_nonconv_model(windowsize,bat,w1,w2,wm1):
  li=layers.Input((windowsize,1))
  x=layers.BatchNormalization(name='bat_layer')(li)
  x=layers.Flatten()(x)
  x=layers.Dropout(0.2)(x)
  output=layers.Dense(100,name='dense_layer1',kernel_constraint=MaskWeights(wm1))(x)
  output=layers.Dense(1,activation='sigmoid',name='dense_layer2')(output)
  model=models.Model(inputs=li, outputs=output)
  for layer_name, initial in [("conv_layer1", locals().get("lconv1")),
                              ("conv_layer2", locals().get("lconv2")),
                              ("bat_layer", locals().get("bat")),
                              ("dense_layer1", locals().get("w1")),
                              ("dense_layer2", locals().get("w2"))]:
    if initial is not None:
      model.get_layer(layer_name).set_weights(initial)
  if "wm1" in locals():
    sparse = model.get_layer("dense_layer1")
    kernel, bias = sparse.get_weights()
    sparse.set_weights([kernel * wm1, bias])
  return model


SETDNNmodel=[]
SETDNNhis=[]
start_time = time.time()
for i in range(4):
  SETDNNhis.append({})
  [noPar1, wm1] = createWeightsMask(epsilon,600,100)
  w1=None
  w2=None
  bat=None
  SETDNNmodel.append(create_SET_nonconv_model(600,bat,w1,w2,wm1))
  for j in range(SET_DNN_EPOCHS):

    SETDNNmodel[i].compile(loss='mean_squared_error', optimizer='sgd', metrics=['accuracy','mean_absolute_error',tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.Precision(name='precision')])
    historytemp = SETDNNmodel[i].fit(trainx,lbs[i],epochs=1)
    for key in historytemp.history:
      
      if key in SETDNNhis[i]:
        SETDNNhis[i][key].append(historytemp.history[key])
      else:
        SETDNNhis[i][key]=[historytemp.history[key]]
    if j + 1 < SET_DNN_EPOCHS:
      bat,w1,w2,wm1=weightsEvolution(SETDNNmodel[i])
      keras.backend.clear_session()
      
      SETDNNmodel[i]=create_SET_nonconv_model(600,bat,w1,w2,wm1)
  
    

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
#normal 1d CONV 
def create_model(windowsize):
  li=layers.Input((windowsize,1))
  x=layers.Conv1D(64, 2, activation='relu')(li)
  x=layers.Conv1D(64, 2, activation='relu')(x)
  x=layers.BatchNormalization()(x)
  x=layers.MaxPooling1D(pool_size=2)(x)
  x=layers.Flatten()(x)
  x=layers.Dropout(0.2)(x)
  output=layers.Dense(100)(x)
  output=layers.Dense(1,activation='sigmoid')(output)
  model=models.Model(inputs=li, outputs=output)
  return model

class TestCallback(keras.callbacks.Callback):
  def __init__(self, test_data):
      self.test_data = test_data
  def on_epoch_end(self, epoch, logs=None):
      x, y = self.test_data
      print(x.shape)
      print(y.shape)
      self.test_time = time.time()
      result = self.model.evaluate(x, y, verbose=1)
      testt=time.time() - self.test_time
      print('\nTesting loss: {}, acc: {}\n'.format(result[0], result[1]))
      print('\nTesting time: {}'.format(testt))
tmodel=[]
tmodel.append(create_model(600))
tmodel.append(create_model(600))
tmodel.append(create_model(600))
tmodel.append(create_model(600))
CONVhis=[]
start_time = time.time()
for i in range(4):

  tmodel[i].compile(optimizer='sgd',loss='mean_squared_error',metrics= ['accuracy','mean_absolute_error',tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.Precision(name='precision')])
  CONVhis.append(tmodel[i].fit(trainx,lbs[i],epochs=EPOCHS,callbacks=[TestCallback((testx, testylis[i]))]))

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
#lasso 1d CONV 
lambda_val=0.7
def create_conv_lasso_model(windowsize):
  li=layers.Input((windowsize,1))
  x=layers.Conv1D(64, 2, activation='relu')(li)
  x=layers.Conv1D(64, 2, activation='relu')(x)
  x=layers.BatchNormalization()(x)
  x=layers.MaxPooling1D(pool_size=2)(x)
  x=layers.Flatten()(x)
  x=layers.Dropout(0.2)(x)
  output=layers.Dense(100)(x)
  output=layers.Dense(1,activation='sigmoid',kernel_regularizer = l1(lambda_val))(output)
  model=models.Model(inputs=li, outputs=output)
  return model


tlasmodel=[]
tlasmodel.append(create_conv_lasso_model(600))
tlasmodel.append(create_conv_lasso_model(600))
tlasmodel.append(create_conv_lasso_model(600))
tlasmodel.append(create_conv_lasso_model(600))
CONVlashis=[]
start_time = time.time()
for i in range(4):

  tlasmodel[i].compile(optimizer='adam',loss='mean_squared_error',metrics= ['accuracy','mean_absolute_error',tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.Precision(name='precision')])
  CONVlashis.append(tlasmodel[i].fit(trainx,lbs[i],epochs=EPOCHS,callbacks=[TestCallback((testx, testylis[i]))]))

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
#group lasso 1d CONV 
l21v=0.002
def create_conv_lasso_model(windowsize):
  li=layers.Input((windowsize,1))
  x=layers.Conv1D(64, 2, activation='relu')(li)
  x=layers.Conv1D(64, 2, activation='relu')(x)
  x=layers.BatchNormalization()(x)
  x=layers.MaxPooling1D(pool_size=2)(x)
  x=layers.Flatten()(x)
  x=layers.Dropout(0.2)(x)
  output=layers.Dense(100)(x)
  output=layers.Dense(20,activation='sigmoid',kernel_regularizer=L21(0.002))(output)
  output=layers.Dense(15,activation='sigmoid',kernel_regularizer=L21(0.002))(output)
  output=layers.Dense(1,activation='sigmoid',kernel_regularizer=L21(0.002))(output)
  model=models.Model(inputs=li, outputs=output)
  return model


cnngrouplasmodel=[]
cnngrouplasmodel.append(create_conv_lasso_model(600))
cnngrouplasmodel.append(create_conv_lasso_model(600))
cnngrouplasmodel.append(create_conv_lasso_model(600))
cnngrouplasmodel.append(create_conv_lasso_model(600))
CONVgrouplashis=[]
start_time = time.time()
for i in range(4):

  cnngrouplasmodel[i].compile(optimizer='adam',loss='mean_squared_error',metrics= ['accuracy','mean_absolute_error',tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.Precision(name='precision')])
  CONVgrouplashis.append(cnngrouplasmodel[i].fit(trainx,lbs[i],epochs=EPOCHS,callbacks=[TestCallback((testx, testylis[i]))]))

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
# All models below use the same training/test split.


In [ ]:
#SET CONV CNN
class TimeHistory(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, batch, logs=None):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, batch, logs=None):
        self.times.append(time.time() - self.epoch_time_start)
import gc
def weightsEvolution(model):
        # this represents the core of the SET procedure. It removes the weights closest to zero in each layer and add new random weights
    lconv1=model.get_layer("conv_layer1").get_weights()
    lconv2=model.get_layer("conv_layer2").get_weights()
    bat=model.get_layer("bat_layer").get_weights()
    w2=model.get_layer("dense_layer2").get_weights()
    w1 = model.get_layer("dense_layer1").get_weights()
    
    [wm1, wm1Core] = rewireMask(w1[0], noPar1)

    w1[0] = w1[0] * wm1Core
    return lconv1,lconv2,bat,w1,w2,wm1

def create_SET_conv_model(windowsize,lconv1,lconv2,bat,w1,w2,wm1):
  li=layers.Input((windowsize,1))
  x=layers.Conv1D(64, 2, activation='relu',name='conv_layer1')(li)
  x=layers.Conv1D(64, 2, activation='relu',name='conv_layer2')(x)
  x=layers.BatchNormalization(name='bat_layer')(x)
  x=layers.MaxPooling1D(pool_size=2)(x)
  x=layers.Flatten()(x)
  x=layers.Dropout(0.2)(x)
  output=layers.Dense(100,name='dense_layer1',kernel_constraint=MaskWeights(wm1))(x)
  output=layers.Dense(1,activation='sigmoid',name='dense_layer2')(output)
  model=models.Model(inputs=li, outputs=output)
  for layer_name, initial in [("conv_layer1", locals().get("lconv1")),
                              ("conv_layer2", locals().get("lconv2")),
                              ("bat_layer", locals().get("bat")),
                              ("dense_layer1", locals().get("w1")),
                              ("dense_layer2", locals().get("w2"))]:
    if initial is not None:
      model.get_layer(layer_name).set_weights(initial)
  if "wm1" in locals():
    sparse = model.get_layer("dense_layer1")
    kernel, bias = sparse.get_weights()
    sparse.set_weights([kernel * wm1, bias])
  return model

SETCONVmodel=[]
SETCONVhis=[]

for i in range(4):
  SETCONVhis.append({})
  [noPar1, wm1] = createWeightsMask(epsilon,19136,100)
  lconv1=None
  lconv2=None
  w1=None
  w2=None
  bat=None
  SETCONVmodel.append(create_SET_conv_model(600,lconv1,lconv2,bat,w1,w2,wm1))
  for j in range(SET_EPOCHS):
    time_callback = TimeHistory()
    
    SETCONVmodel[i].compile(loss='mean_squared_error', optimizer='sgd', metrics=['accuracy','mean_absolute_error',tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.Precision(name='precision')])
    start_time = time.time()
    historytemp = SETCONVmodel[i].fit(trainx,lbs[i],epochs=1,callbacks=[TestCallback((testx, testylis[i])),time_callback])
    end_time = time.time()
    times = time_callback.times
    print("--- THE REAL %s seconds ---" % (times))
    for key in historytemp.history:
      
      if key in SETCONVhis[i]:
        SETCONVhis[i][key].append(historytemp.history[key])
      else:
        SETCONVhis[i][key]=[historytemp.history[key]]
    if j + 1 < SET_EPOCHS:
      lconv1,lconv2,bat,w1,w2,wm1=weightsEvolution(SETCONVmodel[i])
      keras.backend.clear_session()
      gc.collect()
  
      SETCONVmodel[i]=create_SET_conv_model(600,lconv1,lconv2,bat,w1,w2,wm1)

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
#temp SET lasso CONV CNN
class TimeHistory(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, batch, logs=None):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, batch, logs=None):
        self.times.append(time.time() - self.epoch_time_start)
import gc
def weightsEvolution(model):
        # this represents the core of the SET procedure. It removes the weights closest to zero in each layer and add new random weights
    lconv1=model.get_layer("conv_layer1").get_weights()
    lconv2=model.get_layer("conv_layer2").get_weights()
    bat=model.get_layer("bat_layer").get_weights()
    w2=model.get_layer("dense_layer2").get_weights()
    w1 = model.get_layer("dense_layer1").get_weights()
    
    [wm1, wm1Core] = rewireMask(w1[0], noPar1)

    w1[0] = w1[0] * wm1Core
    return lconv1,lconv2,bat,w1,w2,wm1
lambda_val=0.7
def create_SET_lasso_conv_model(windowsize,lconv1,lconv2,bat,w1,w2,wm1):
  li=layers.Input((windowsize,1))
  x=layers.Conv1D(64, 2, activation='relu',name='conv_layer1')(li)
  x=layers.Conv1D(64, 2, activation='relu',name='conv_layer2')(x)
  x=layers.BatchNormalization(name='bat_layer')(x)
  x=layers.MaxPooling1D(pool_size=2)(x)
  x=layers.Flatten()(x)
  x=layers.Dropout(0.2)(x)
  output=layers.Dense(100,name='dense_layer1',kernel_constraint=MaskWeights(wm1))(x)
  output=layers.Dense(1,activation='sigmoid',name='dense_layer2',kernel_regularizer = l1(lambda_val))(output)
  model=models.Model(inputs=li, outputs=output)
  for layer_name, initial in [("conv_layer1", locals().get("lconv1")),
                              ("conv_layer2", locals().get("lconv2")),
                              ("bat_layer", locals().get("bat")),
                              ("dense_layer1", locals().get("w1")),
                              ("dense_layer2", locals().get("w2"))]:
    if initial is not None:
      model.get_layer(layer_name).set_weights(initial)
  if "wm1" in locals():
    sparse = model.get_layer("dense_layer1")
    kernel, bias = sparse.get_weights()
    sparse.set_weights([kernel * wm1, bias])
  return model

SETLASSOCONVmodel=[]
SETLASSOCONVhis=[]

for i in range(4):
  SETLASSOCONVhis.append({})
  [noPar1, wm1] = createWeightsMask(epsilon,19136,100)
  lconv1=None
  lconv2=None
  w1=None
  w2=None
  bat=None
  SETLASSOCONVmodel.append(create_SET_lasso_conv_model(600,lconv1,lconv2,bat,w1,w2,wm1))
  for j in range(SET_EPOCHS):
    time_callback = TimeHistory()
    
    SETLASSOCONVmodel[i].compile(loss='mean_squared_error', optimizer='adam', metrics=['accuracy','mean_absolute_error',tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.Precision(name='precision')])
    start_time = time.time()
    historytemp = SETLASSOCONVmodel[i].fit(trainx,lbs[i],epochs=1,callbacks=[TestCallback((testx, testylis[i])),time_callback])
    end_time = time.time()
    times = time_callback.times
    print("--- THE REAL %s seconds ---" % (times))
    for key in historytemp.history:
      
      if key in SETLASSOCONVhis[i]:
        SETLASSOCONVhis[i][key].append(historytemp.history[key])
      else:
        SETLASSOCONVhis[i][key]=[historytemp.history[key]]
    if j + 1 < SET_EPOCHS:
      lconv1,lconv2,bat,w1,w2,wm1=weightsEvolution(SETLASSOCONVmodel[i])
      keras.backend.clear_session()
      gc.collect()
  
      SETLASSOCONVmodel[i]=create_SET_lasso_conv_model(600,lconv1,lconv2,bat,w1,w2,wm1)

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
# Flatten the legacy SET history representation for consistent plotting/export.
for collection in (SETDNNhis, SETCONVhis, SETLASSOCONVhis):
    for history in collection:
        for key, values in history.items():
            history[key] = np.asarray(values).reshape(-1).tolist()
dh = [h.history for h in DNNhis]
sdh = SETDNNhis
ch = [h.history for h in CONVhis]
sch = SETCONVhis


In [ ]:
bar_width = 0.35
opacity = 0.8
a2,b=createWeightsMask(epsilon,19136,100)
index = np.arange(4)

fig, ax = plt.subplots()

def countWeight(model):
  weight_count=[]
  for var in model.trainable_variables:
    weight_count.append(tf.math.count_nonzero(var))
  return np.sum(weight_count)


conv_weight=countWeight(tmodel[0])
las_conv_weight=countWeight(tlasmodel[0])
group_conv_weight=countWeight(cnngrouplasmodel[0])
set_conv_weight=countWeight(SETCONVmodel[0])
set_LASSO_conv_weight=countWeight(SETLASSOCONVmodel[0])

rects1 = plt.bar(0, conv_weight, bar_width,
alpha=opacity,
color='red',
label='CONV')

rects2 = plt.bar(bar_width, las_conv_weight, bar_width,
alpha=opacity,
color='g',
label='LASSO CONV')


rects3 = plt.bar(bar_width*2, group_conv_weight, bar_width,
alpha=opacity,
color='b',
label='GROUP LASSO CONV')

rects4 = plt.bar(bar_width*3, set_conv_weight, bar_width,
alpha=opacity,
color='y',
label='SET CONV')

rects4 = plt.bar(bar_width*4, set_LASSO_conv_weight, bar_width,
alpha=opacity,
color='m',
label='LASSO SET CONV')

plt.ylabel('weights number')
plt.title('weights number graph')
#plt.xticks(index + bar_width, ('washing machine', 'dish washer', 'fridage', 'micromwave'))
plt.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / ('conv_weights.png'), dpi=150)
plt.show(); plt.close()

In [ ]:
gn=['loss','accuracy']
device=['washing machine', 'dish washer', 'fridge', 'micromwave']

for i in range(2):
  for j in range(4):
    index=range(len(CONVhis[j].history[gn[i]]))
    plt.plot(range(len(CONVhis[j].history[gn[i]])),CONVhis[j].history[gn[i]],label='CONV')
    plt.plot(range(len(CONVlashis[j].history[gn[i]])),CONVlashis[j].history[gn[i]], label='LASSO CONV')
    plt.plot(range(len(CONVgrouplashis[j].history[gn[i]])),CONVgrouplashis[j].history[gn[i]],label='GROUP LASSO CONV')
    plt.plot(range(len(SETCONVhis[j][gn[i]])),SETCONVhis[j][gn[i]],label='SET CONV')
    plt.plot(range(len(SETLASSOCONVhis[j][gn[i]])),SETLASSOCONVhis[j][gn[i]],label='LASSO SET CONV')
    plt.xlabel(device[j])
    plt.ylabel('value')
    plt.title(gn[i].upper()+' CURVE')
    plt.legend();
    plt.savefig(OUTPUT_DIR / ('conv_' + device[j]+' '+gn[i]+' curve'+'.png'), dpi=150)
    plt.show(); plt.close()

## Plot results


In [ ]:
ms=['DNN','SET DNN','CONV','SET CONV']
device=['washing machine', 'dish washer', 'fridge', 'micromwave']
paras=['loss','accuracy','mean_absolute_error','recall','precision']
bar_width = 0.35
opacity = 0.8
for i in range(4):
  for j in range(5):

    index = np.arange(4)

    fig, ax = plt.subplots()

    rects1 = plt.bar(0, dh[i][paras[j]][-1], bar_width,
    alpha=opacity,
    color='red',
    label='DNN')

    rects2 = plt.bar(bar_width, sdh[i][paras[j]][-1], bar_width,
    alpha=opacity,
    color='g',
    label='SET DNN')

    rects3 = plt.bar(bar_width*2, ch[i][paras[j]][-1], bar_width,
    alpha=opacity,
    color='b',
    label='CONV')

    rects4 = plt.bar(bar_width*3, sch[i][paras[j]][-1], bar_width,
    alpha=opacity,
    color='y',
    label='SET CONV')

    plt.xlabel(device[i])
    plt.ylabel('value')
    plt.title(paras[j]+' graph')
    #plt.xticks(index + bar_width, ('washing machine', 'dish washer', 'fridage', 'micromwave'))
    plt.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / (device[i]+' '+paras[j]+' graph'+'.png'), dpi=150)
    plt.show(); plt.close()

In [ ]:
for j in range(4):
  with open('SETCONVhis'+str(j), 'wb') as file_pi:
        pickle.dump(SETCONVhis[j], file_pi)


In [ ]:
import pickle
for j in range(4):
  with open('DNNh'+str(j), 'wb') as file_pi:
        pickle.dump(DNNhis[j].history, file_pi)
  with open('SETDNNh'+str(j), 'wb') as file_pi:
        pickle.dump(SETDNNhis[j], file_pi)
  with open('CNNh'+str(j), 'wb') as file_pi:
        pickle.dump(CONVhis[j].history, file_pi)

In [ ]:
import pickle
dh=[0]*4
sdh=[0]*4
ch=[0]*4
sch=[0]*4
for j in range(4):
  dh[j] = pickle.load(open('DNNh'+str(j), "rb"))
  sdh[j] = pickle.load(open('SETDNNh'+str(j), "rb"))
  ch[j] = pickle.load(open('CNNh'+str(j), "rb"))
  sch[j] = pickle.load(open('SETCONVhis'+str(j), "rb"))

In [ ]:
gn=['loss','accuracy']
device=['washing machine', 'dish washer', 'fridge', 'micromwave']

for i in range(2):
  for j in range(4):
    index=range(len(dh[j][gn[i]]))
    plt.plot(range(len(dh[j][gn[i]])),dh[j][gn[i]],label='DNN')
    plt.plot(range(len(sdh[j][gn[i]])),sdh[j][gn[i]], label='SET DNN')
    plt.plot(range(len(ch[j][gn[i]])),ch[j][gn[i]],label='CONV')
    plt.plot(range(len(sch[j][gn[i]])),sch[j][gn[i]],label='SET CONV')
    plt.xlabel(device[j])
    plt.ylabel('value')
    plt.title(gn[i]+' curve')
    plt.legend();
    plt.savefig(OUTPUT_DIR / (device[j]+' '+gn[i]+' curve'+'.png'), dpi=150)
    plt.show(); plt.close()

In [ ]:
# Check finite training histories, held-out metrics, and sparse-mask enforcement.
import csv
families = {
    'DNN': (DNNmodel, [h.history for h in DNNhis]),
    'LASSO DNN': (DNNlasmodel, [h.history for h in DNNlashis]),
    'GROUP LASSO DNN': (DNNgrouplasmodel, [h.history for h in DNNgrouplashis]),
    'SET DNN': (SETDNNmodel, SETDNNhis),
    'CONV': (tmodel, [h.history for h in CONVhis]),
    'LASSO CONV': (tlasmodel, [h.history for h in CONVlashis]),
    'GROUP LASSO CONV': (cnngrouplasmodel, [h.history for h in CONVgrouplashis]),
    'SET CONV': (SETCONVmodel, SETCONVhis),
    'LASSO SET CONV': (SETLASSOCONVmodel, SETLASSOCONVhis),
}
rows = []
for family, (model_list, histories) in families.items():
    for target, (model, history) in enumerate(zip(model_list, histories)):
        assert all(np.isfinite(values).all() for values in history.values())
        scores = model.evaluate(testx, testylis[target], verbose=0, return_dict=True)
        assert all(np.isfinite(value) for value in scores.values())
        if 'SET' in family:
            layer = model.get_layer('dense_layer1')
            assert np.all(layer.get_weights()[0][layer.kernel_constraint.mask == 0] == 0)
        rows.append(dict(mode=RUN_MODE, family=family, target=target, **scores))
with open(OUTPUT_DIR / 'heldout_metrics.csv', 'w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
# Exercise rewiring and custom-object serialization independently.
_, sample_mask = createWeightsMask(2, 30, 10)
sample_weights = np.random.normal(size=sample_mask.shape).astype('float32') * sample_mask
new_mask, core = rewireMask(sample_weights, sample_mask.sum())
assert new_mask.sum() == sample_mask.sum()
assert np.all(core <= new_mask)
assert np.any(new_mask != sample_mask)
assert keras.regularizers.deserialize(keras.regularizers.serialize(L21(.001))).C == .001
print(f'Validated {len(rows)} model/target combinations in {RUN_MODE} mode.')
